# PCA / Dimensionality Reduction

Runs Principal Component Analysis on selected numeric columns and adds `PC1#number`, `PC2#number`, ... to the survey. Optionally adds a 2D UMAP projection (`UMAP1#number`, `UMAP2#number`) for nonlinear embedding.

In [ ]:
# -- Colab bootstrap (no-op on Binder / JupyterHub) ------------------------
import os, sys
if "google.colab" in sys.modules:
    import subprocess
    if os.path.isdir("/tmp/suave-nb/.git"):
        subprocess.run(["git","-C","/tmp/suave-nb","fetch","--depth=1","origin","main"],
                       capture_output=True)
        subprocess.run(["git","-C","/tmp/suave-nb","reset","--hard","origin/main"],
                       capture_output=True)
    else:
        _r = subprocess.run(
            ["git","clone","--depth=1",
             "https://github.com/izaslavsky/suave-notebooks.git","/tmp/suave-nb"],
            capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError(f"Could not clone suave-notebooks:\n{_r.stderr}")
    sys.path.insert(0, "/tmp/suave-nb/helpers")


In [ ]:
# ── Colab only: mount Google Drive to load session credentials ─────────────
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# ── Load SuAVE parameters ──────────────────────────────────────────────────
import sys, os, requests as _req, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

if not su.ENV.colab:
    # Binder / JupyterHub: parameters load automatically
    _p = su.load_params()
    if _p is None:
        from IPython.display import HTML as _HTML
        display(_HTML(
            '<div style="background:#dc3545;color:white;padding:14px 16px;'
            'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
            '&#9888; No SuAVE parameters found.'
            '<br><span style="font-size:12px;font-weight:normal">'
            'Open this notebook via SuAVEDispatch from your survey in SuAVE.'
            '</span></div>'))
        class _Stop(Exception):
            def _render_traceback_(self): return []
        raise _Stop()
else:
    # Colab: try Drive/cache silently; credentials cell below handles fallback
    _p = su.load_params(_silent=True)


In [ ]:
# -- Colab: verify session loaded from Drive/cache --------------------------
from IPython.display import HTML
if su.ENV.colab:
    if _p is None:
        display(HTML(
            '<div style="background:#dc3545;color:white;padding:14px 16px;'
            'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
            '&#9888; No SuAVE session found.'
            '<br><span style="font-size:12px;font-weight:normal">'
            'Open SuAVEDispatch from the correct survey in SuAVE, then re-open this notebook.'
            '</span></div>'))
        class _Stop(Exception):
            def _render_traceback_(self): return []
        raise _Stop()
    display(HTML(
        '<div style="font-size:12px;padding:8px 12px;border-radius:4px;margin:3px 0;'
        'background:#e8f5e9;border:1px solid #81c784">'
        'Session loaded &mdash; survey <b>' + _p.get('survey', '?') + '</b>'
        ', user <b>' + _p.get('user', '?') + '</b>.'
        '<br><span style="color:#666;font-size:11px">'
        'Wrong survey? Go back to SuAVE, open the correct survey, and click Jupyter again.'
        '</span></div>'))
else:
    su._skipped('Colab only')


In [ ]:
# ── Variables used throughout this notebook ────────────────────────────────
if _p is None:
    from IPython.display import HTML as _HTML
    display(_HTML(
        '<div style="background:#dc3545;color:white;padding:14px 16px;'
        'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
        '&#9888; Parameters not loaded.'
        '<br><span style="font-size:12px;font-weight:normal">'
        'Run the credentials cell above, then re-run this cell.'
        '</span></div>'))
    class _Stop(Exception):
        def _render_traceback_(self): return []
    raise _Stop()
user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'

localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images

absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)

su.show_params(_p)

In [ ]:
import sys
sys.path.insert(1, '../../helpers')
import panel_libs as panellibs
import suave_integration as suaveint

import ipywidgets as widgets
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

## 1. Load data and select numeric columns

In [ ]:
df = panellibs.extract_data(absolutePath + csv_file)
print(f"Loaded {len(df)} rows")

num_cols = [c for c in df.columns if '#number' in c]
if not num_cols:
    raise ValueError('No #number columns found in this survey.')

col_selector = widgets.SelectMultiple(
    options=num_cols, value=num_cols[:min(5, len(num_cols))],
    description='Columns:', rows=min(10, len(num_cols)),
    layout=widgets.Layout(width='70%', height='200px')
)
n_components = widgets.IntSlider(
    value=3, min=2, max=min(10, len(num_cols)),
    description='# PCs:', continuous_update=False
)
umap_toggle = widgets.Checkbox(value=False, description='Also compute UMAP 2D embedding')

printmd("**Select columns and number of principal components:**")
display(col_selector, n_components, umap_toggle)

## 2. Run PCA

In [ ]:
selected = list(col_selector.value)
n_comp   = n_components.value

X = df[selected].apply(pd.to_numeric, errors='coerce')
missing = X.isnull().sum().sum()
if missing > 0:
    printmd(f"**{missing} missing values imputed with column mean.**")
    X = X.fillna(X.mean())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=n_comp)
coords = pca.fit_transform(X_scaled)

for i in range(n_comp):
    df[f'PC{i+1}#number'] = coords[:, i].round(6)

# Scree plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
explained = pca.explained_variance_ratio_ * 100
axes[0].bar(range(1, n_comp + 1), explained)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance explained (%)')
axes[0].set_title('Scree plot')

# Loading biplot (PC1 vs PC2)
short_names = [c.split('#')[0] for c in selected]
for j, name in enumerate(short_names):
    axes[1].arrow(0, 0, pca.components_[0, j], pca.components_[1, j],
                  head_width=0.02, color='steelblue')
    axes[1].text(pca.components_[0, j] * 1.1, pca.components_[1, j] * 1.1,
                 name, fontsize=8)
axes[1].set_xlim(-1.2, 1.2); axes[1].set_ylim(-1.2, 1.2)
axes[1].set_title('Loading biplot (PC1 vs PC2)')
axes[1].axhline(0, color='gray', lw=0.5); axes[1].axvline(0, color='gray', lw=0.5)
plt.tight_layout()
plt.show()

printmd(f"**PC1–PC{n_comp} explain {explained.sum():.1f}% of variance.**")

## 3. Optional UMAP embedding

In [ ]:
if umap_toggle.value:
    try:
        import umap
    except ImportError:
        print('Installing umap-learn...')
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'umap-learn'])
        import umap

    reducer = umap.UMAP(n_components=2, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    df['UMAP1#number'] = embedding[:, 0].round(6)
    df['UMAP2#number'] = embedding[:, 1].round(6)

    plt.figure(figsize=(7, 5))
    plt.scatter(embedding[:, 0], embedding[:, 1], s=10, alpha=0.5)
    plt.title('UMAP 2D embedding')
    plt.xlabel('UMAP1'); plt.ylabel('UMAP2')
    plt.tight_layout()
    plt.show()
    printmd('**UMAP1#number and UMAP2#number added.**')
else:
    printmd('*UMAP not requested — skipping.*')

## 4. Save and publish to SuAVE

In [ ]:
new_file = suaveint.save_csv_file(df, absolutePath, csv_file)

In [ ]:
#Input survey name
survey_name = input('Enter survey name: ')
printmd('**Survey Name:** ' + survey_name)
